In [ ]:
#埋め込みの読み込み
import numpy as np
from gensim.models import KeyedVectors

def load_pretrained_embeddings(embedding_path, vocab_limit=None):
    embeddings = []
    token2id = {}
    id2token = {}

    # GoogleNews-vectors-negative300.bin を読み込む
    word_vectors = KeyedVectors.load_word2vec_format(embedding_path, binary=True)

    embedding_dim = word_vectors.vector_size
    embeddings.append(np.zeros(embedding_dim))  # <PAD>トークン用ゼロベクトル
    token2id['<PAD>'] = 0
    id2token[0] = '<PAD>'

    for idx, word in enumerate(word_vectors.index_to_key):
        if vocab_limit and (idx >= vocab_limit):
            break
        vector = word_vectors[word]

        token_id = len(embeddings)
        token2id[word] = token_id
        id2token[token_id] = word
        embeddings.append(vector)

    embedding_matrix = np.vstack(embeddings) #各単語の埋め込みを縦方向に結合(ベクトル→行列)
    return embedding_matrix, token2id, id2token

# 使い方
embedding_path = 'GoogleNews-vectors-negative300.bin'
embedding_matrix, token2id, id2token = load_pretrained_embeddings(embedding_path, vocab_limit=50000)


In [ ]:
#データセットの読み込み
import csv
import torch

def load_sst_data(file_path, token2id):
    dataset = []
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            text = row['sentence']
            label = int(row['label'])

            # 単語ごとにトークンIDに変換
            tokens = text.split()
            input_ids = [token2id[token] for token in tokens if token in token2id]

            # 全部消えたらスキップ
            if len(input_ids) == 0:
                continue

            item = {
                'text': text,
                'label': torch.tensor([float(label)]),
                'input_ids': torch.tensor(input_ids)
            }
            dataset.append(item)
    return dataset

# 使い方
train_file = './SST-2/train.tsv'
dev_file = './SST-2/dev.tsv'

train_data = load_sst_data(train_file, token2id)
dev_data = load_sst_data(dev_file, token2id)


In [ ]:
#モデルの構築
import torch
import torch.nn as nn

class TextAverageLogisticRegression(nn.Module):
    def __init__(self, embedding_matrix):
        super(TextAverageLogisticRegression, self).__init__()
        vocab_size, embedding_dim = embedding_matrix.shape

        # 埋め込み層
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
        self.embedding.weight.requires_grad = False  # 事前学習済みなので凍結

        # ロジスティック回帰（線形層）
        self.linear = nn.Linear(embedding_dim, 1)

    def forward(self, input_ids):
        """
        input_ids: (バッチサイズ, シーケンス長)
        """
        embedded = self.embedding(input_ids)  # (バッチサイズ, シーケンス長, 埋め込み次元)
        
        # マスク：IDが0（PAD）なら無視
        mask = (input_ids != 0).unsqueeze(-1)  # (バッチサイズ, シーケンス長, 1)
        masked_embeddings = embedded * mask

        # 平均計算（マスクした上で）
        summed = masked_embeddings.sum(dim=1)  # (バッチサイズ, 埋め込み次元)
        lengths = mask.sum(dim=1)  # (バッチサイズ, 1)
        avg_embedded = summed / lengths.clamp(min=1)  # 長さ0回避

        logits = self.linear(avg_embedded)  # (バッチサイズ, 1)
        probs = torch.sigmoid(logits)  # (バッチサイズ, 1)
        return probs
